In [1]:
class GPT:
    def __init__(self,num_dims,num_heads,num_layers,vocab_size,seq_len):
        self.seq_len = seq_len
        self.wte = TokenEmbeddings(vocab_size,num_dims)
        self.wpe = PositionalEmbeddings(seq_len,num_dims)

        self.blocks = [Transformer(num_dims,num_heads) for _ in range(num_layers)]
        
        self.ln_f = LayerNorm(num_dims)
        self.lm_head = LinearLayer(num_dims,vocab_size)

    def forward(self,idx):
        B,T = idx.shape

        pos = torch.arange(T, device=idx.device)

        tok_emb = self.wte.forward(idx)
        pos_emb = self.wpe.forward(pos)

        x = tok_emb+pos_emb

        for block in self.blocks:
            x = block.forward(x)

        x  = self.ln_f.forward(x)
        logits = self.lm_head.forward(x)

        return logits


  def backward(self, grad_out):
      
        grad_logits = self.lm_head.backward(grad_out)
        grad_ln = self.ln_f.backward(grad_logits)


        grad = grad_ln
        for block in reversed(self.blocks):
            grad = block.backward(grad)

        self.wpe.backward(grad)
        self.wte.backward(grad)

        return grad
        
    

every module to this part are an individual paid actors 😂️
now we will bring all of them into the single show 

as you can see below the order of flow 
we will instantiate the classes we have written above and after that we can start the flow


### 1. What the Whole Assembly Line Looks Like

```
  ├─> wte (Token Embeddings) ─────┐
  │                               ▼
  └─> wpe (Position Embeddings) ─> (+) = x : (B, T, d_model)
                                   │
                                   ▼
                        ┌─────────────────────┐
                        │ TransformerBlock 1  │
                        └──────────┬──────────┘
                                   ▼
                        ┌─────────────────────┐
                        │ TransformerBlock 2  │
                        └──────────┬──────────┘
                                   ▼
                                 [ ... ]
                                   ▼
                        ┌─────────────────────┐
                        │ TransformerBlock N  │
                        └──────────┬──────────┘
                                   │
                                   ▼
                              Final LN (ln_f)
                                   │
                                   ▼
                            LM Head (Linear)
                                   │
                                   ▼
                          Logits: (B, T, vocab_size)

```

we are at the gpt class right
and then,we are gonna instantiate the objects for the token embeddings and then positional embeddings too 

and after that we need to create the stack of the Transformers,using the list comprehension
and then we need the two final objects they are
1)final linera layer so this can actually set right the entire calculations
2)the ultimate lm heads,here we start gen the tokens,so this now takes the dims and then look them into the vocubulary we have 

we will bring the idices and then we will use the foreard method here
ad we take the shep and index 
